# Snowpark Container Services (SPCS) — Complete Guide

This notebook is a comprehensive, beginner-friendly guide to **Snowpark Container Services (SPCS)**. It covers every component you need to understand, from architecture to deployment to monitoring.

---

## What is SPCS?

Snowpark Container Services is a **fully managed container orchestration platform** inside Snowflake. It lets you run any containerized application (Docker containers) directly within your Snowflake account — no external Kubernetes cluster needed.

**Why does it exist?**
- Snowflake SQL is great for queries, but not for custom ML models, web apps, or non-SQL logic.
- Snowpark (Python/Java/Scala) helps, but is limited to Snowflake's runtime.
- SPCS gives you **full control** — run ANY code, ANY language, ANY framework, with access to Snowflake data.

**Common use cases:**
- Hosting ML models for inference (e.g., sentiment analysis, image classification)
- Running web applications / APIs over Snowflake data
- Batch data processing jobs
- Running GPU workloads (model training, LLMs)
- Hosting custom microservices

---
## Architecture Overview

Here's how all the pieces fit together:

```
┌─────────────────────────────────────────────────────────────────────┐
│                        SNOWFLAKE ACCOUNT                            │
│                                                                     │
│  ┌──────────────────┐       ┌──────────────────────────────────┐   │
│  │  IMAGE REPOSITORY │       │         COMPUTE POOL              │   │
│  │                  │       │  ┌────────────┐ ┌────────────┐   │   │
│  │  Your Docker     │──────▶│  │   Node 0   │ │   Node 1   │   │   │
│  │  images live     │       │  │            │ │            │   │   │
│  │  here            │       │  │ ┌────────┐ │ │ ┌────────┐ │   │   │
│  └──────────────────┘       │  │ │Service │ │ │ │Service │ │   │   │
│                             │  │ │Instance│ │ │ │Instance│ │   │   │
│                             │  │ └────────┘ │ │ └────────┘ │   │   │
│                             │  └────────────┘ └────────────┘   │   │
│                             └──────────────────────────────────┘   │
│                                        │                           │
│                                        ▼                           │
│  ┌──────────────────┐       ┌──────────────────┐                   │
│  │ SERVICE FUNCTION  │◀─────│    ENDPOINTS      │                   │
│  │ (SQL UDF)        │       │ (HTTP ports)      │                   │
│  └──────────────────┘       └──────────────────┘                   │
│           │                          │                             │
│           ▼                          ▼                             │
│  ┌──────────────────┐       ┌──────────────────┐                   │
│  │  SQL Queries      │       │  Internet/Ingress │                   │
│  │  SELECT func(x)   │       │  (public endpoint)│                   │
│  └──────────────────┘       └──────────────────┘                   │
└─────────────────────────────────────────────────────────────────────┘
```

**Data flow:** You push a Docker image → Snowflake runs it in a Compute Pool → You interact with it via Endpoints (HTTP) or Service Functions (SQL).

---
## All Components at a Glance

| # | Component | What It Does |
|---|-----------|-------------|
| 1 | **Image Repository** | Stores your Docker/OCI images inside Snowflake |
| 2 | **Compute Pool** | VM nodes that run your containers |
| 3 | **Service** | A long-running containerized application |
| 4 | **Job Service** | A one-time/batch container that exits when done |
| 5 | **Service Specification** | YAML that tells Snowflake how to run your container |
| 6 | **Endpoints** | Network ports your service exposes (ingress) |
| 7 | **Ingress Gateway** | Authenticates and routes external traffic to public endpoints |
| 8 | **Service Function** | A SQL UDF that calls your service's HTTP endpoint |
| 9 | **External Access Integration** | Allows your container to reach the internet (egress) |
| 10 | **Volumes** | Persistent or shared storage for containers |
| 11 | **Secrets** | Credentials passed securely into containers |
| 12 | **Event Table / Logs** | Where container logs, metrics, and traces are stored |

Let's go through each one in detail.

---
## 1. Image Repository

**What:** A private Docker image registry hosted inside your Snowflake account. It stores your Docker images — the same way Docker Hub or AWS ECR stores images, but it lives within Snowflake.

**What is a Docker image?**

A Docker image is a packaged bundle of your application code + all its dependencies (OS libraries, Python packages, model files, etc.) frozen into a single portable file.

```
Your code (app.py, model.pkl, etc.)
   + Python 3.11
   + Flask, PyTorch, etc.
   + Linux OS layer
   = Docker Image (one portable file)
```

**Key terminology (don't mix these up!):**
- **Image** = the blueprint. A static, read-only template. It defines *what* to run but isn't running anything.
- **Instance (Container)** = the deployed, running application on a compute pool. Created *from* an image.

Same relationship as: Class → Object, Recipe → Dish, Blueprint → Building.

One image can spawn many instances (containers). When you `CREATE SERVICE`, Snowflake takes the image (blueprint) and launches one or more instances (running containers) on the compute pool.

**The flow:**

```
Your laptop                         Snowflake
───────────                         ─────────────────────────
1. Write code (app.py)
2. Write Dockerfile
3. docker build → creates image
4. docker push ──────────────────→  Image Repository (stores it)
                                         ↓
                                    CREATE SERVICE (pulls image from repo)
                                         ↓
                                    Container runs on Compute Pool
```

**Why not use Docker Hub directly?**
- Security — images stay within your Snowflake account boundary
- No internet access needed at runtime — the compute pool pulls from the internal repo
- Access control — Snowflake RBAC governs who can push/pull images
- Required — SPCS **only** pulls images from Snowflake image repositories, not external registries

**Key facts:**
- One repository can hold multiple images with multiple tags
- Uses standard Docker push/pull commands
- Images are stored per-database, per-schema
- Snowflake serves the OCI v2 API (compatible with Docker, Podman, etc.)

In [ ]:
%%sql -r create_repo_result
-- Create an image repository
CREATE DATABASE IF NOT EXISTS spcs_demo_db;
CREATE SCHEMA IF NOT EXISTS spcs_demo_db.spcs_schema;

CREATE IMAGE REPOSITORY IF NOT EXISTS spcs_demo_db.spcs_schema.my_repo;

-- View your image repositories (note the repository_url column — you need this to push images)
SHOW IMAGE REPOSITORIES IN SCHEMA spcs_demo_db.spcs_schema;

**How to push an image (from your local terminal, NOT from this notebook):**

```bash
# 1. Authenticate Docker with Snowflake registry
snow spcs image-registry login

# 2. Build your image for linux/amd64 (SPCS ONLY supports this platform)
docker build --platform linux/amd64 -t <repository_url>/<image_name>:<tag> .

# 3. Push to Snowflake
docker push <repository_url>/<image_name>:<tag>
```

The `repository_url` comes from the SHOW IMAGE REPOSITORIES output above.

**Example:**
```bash
docker build --platform linux/amd64 \
  -t myorg-myacct.registry.snowflakecomputing.com/spcs_demo_db/spcs_schema/my_repo/my_app:v1 .

docker push \
  myorg-myacct.registry.snowflakecomputing.com/spcs_demo_db/spcs_schema/my_repo/my_app:v1
```

**Important:** SPCS only supports `linux/amd64`. If you're on an M1/M2 Mac (ARM), you MUST use `--platform linux/amd64` when building.

---
## 2. Compute Pool

**What:** A collection of virtual machine (VM) nodes where your containers run.

**Why:** Services need compute resources (CPU, memory, GPU). A compute pool provides them.

**Key facts:**
- You choose the machine type (`INSTANCE_FAMILY`) — determines CPU, RAM, GPU
- You set MIN/MAX nodes — Snowflake auto-scales within this range
- Multiple services can share one compute pool
- Compute pools cost credits while active (even if idle with no services)
- They auto-suspend after inactivity (configurable with `AUTO_SUSPEND_SECS`)

**Common instance families:**

| Family | Type | Use Case |
|--------|------|----------|
| `CPU_X64_XS` | CPU, minimal | Tiny services, testing |
| `CPU_X64_S` | CPU, small | Light APIs, web apps |
| `CPU_X64_M` | CPU, medium | Data processing |
| `CPU_X64_L` | CPU, large | Heavy batch jobs |
| `GPU_NV_S` | GPU (NVIDIA) | ML inference |
| `GPU_NV_M` | GPU (NVIDIA) | ML training |
| `GPU_NV_L` | GPU (NVIDIA) | Large model training / LLMs |

In [ ]:
%%sql -r instance_families
-- See what instance families are available in your account/region
SHOW COMPUTE POOL INSTANCE FAMILIES;

-- Create a compute pool
CREATE COMPUTE POOL IF NOT EXISTS spcs_demo_pool
  MIN_NODES = 1
  MAX_NODES = 2
  INSTANCE_FAMILY = CPU_X64_S
  AUTO_RESUME = TRUE
  AUTO_SUSPEND_SECS = 300;  -- suspend after 5 minutes of inactivity

-- Check compute pool status (Wait for ACTIVE or IDLE before creating services)
DESCRIBE COMPUTE POOL spcs_demo_pool;

**Compute pool lifecycle states:**

| State | Meaning |
|-------|--------|
| `STARTING` | Nodes are being provisioned — wait for this to finish |
| `IDLE` | Ready, no services running |
| `ACTIVE` | Services are running on it |
| `STOPPING` | Shutting down |
| `SUSPENDED` | No cost — will auto-resume when a service needs it |
| `RESIZING` | Adding or removing nodes |

**Management commands:**
```sql
ALTER COMPUTE POOL spcs_demo_pool SUSPEND;   -- stop billing
ALTER COMPUTE POOL spcs_demo_pool RESUME;    -- restart
DROP COMPUTE POOL spcs_demo_pool;            -- delete (must drop services first)
```

---
## 3. Service (Long-Running)

**What:** A continuously running containerized application. Like a web server — it runs until you explicitly stop it.

**Why:** For APIs, web UIs, ML inference endpoints, or any always-on workload.

**Key facts:**
- If a container crashes, Snowflake automatically restarts it
- Can run multiple instances (replicas) for high availability and load balancing
- Created with `CREATE SERVICE`
- Stopped with `DROP SERVICE` or `ALTER SERVICE ... SUSPEND`

In [ ]:
%%sql -r create_service_result
-- Create a service using inline YAML specification
CREATE SERVICE IF NOT EXISTS spcs_demo_db.spcs_schema.echo_service
  IN COMPUTE POOL spcs_demo_pool
  FROM SPECIFICATION $$
  spec:
    containers:
    - name: echo
      image: /spcs_demo_db/spcs_schema/my_repo/my_echo_app:latest
      env:
        SERVER_PORT: "8080"
      resources:
        requests:
          memory: 512M
          cpu: 0.5
        limits:
          memory: 1G
          cpu: 1
      readinessProbe:
        port: 8080
        path: /health
    endpoints:
    - name: api
      port: 8080
      public: false
  $$
  MIN_INSTANCES = 1
  MAX_INSTANCES = 1;

-- Check service status
DESCRIBE SERVICE spcs_demo_db.spcs_schema.echo_service;

-- Check individual container status within the service
SHOW SERVICE CONTAINERS IN SERVICE spcs_demo_db.spcs_schema.echo_service;

---
## 4. Job Service (Batch / One-Time)

**What:** A container that runs once and exits. Like a stored procedure but with full container flexibility.

**Why:** For batch processing, model training, ETL jobs, data migrations — anything with a start and end.

**Key differences from a long-running service:**

| | Long-Running Service | Job Service |
|---|---|---|
| Created with | `CREATE SERVICE` | `EXECUTE JOB SERVICE` |
| Lifetime | Runs until stopped | Exits when code finishes |
| Restart on crash | Yes (auto-restart) | No (failure = done) |
| Multiple replicas | For load balancing | For parallel batch work |
| Use case | APIs, web apps | Training, ETL, migration |

In [ ]:
%%sql -r job_result
-- Execute a job service (synchronous — blocks until completion)
EXECUTE JOB SERVICE
  IN COMPUTE POOL spcs_demo_pool
  NAME = spcs_demo_db.spcs_schema.my_batch_job
  FROM SPECIFICATION $$
  spec:
    containers:
    - name: worker
      image: /spcs_demo_db/spcs_schema/my_repo/my_job_image:latest
      env:
        SNOWFLAKE_WAREHOUSE: COMPUTE_WH
      args:
      - "--input_table=my_source_table"
      - "--output_table=my_results"
  $$;

In [ ]:
%%sql -r async_job_result
-- Execute an ASYNC job (returns immediately, runs in background)
EXECUTE JOB SERVICE
  IN COMPUTE POOL spcs_demo_pool
  NAME = spcs_demo_db.spcs_schema.my_async_job
  ASYNC = TRUE
  FROM SPECIFICATION $$
  spec:
    containers:
    - name: trainer
      image: /spcs_demo_db/spcs_schema/my_repo/ml_trainer:latest
      env:
        SNOWFLAKE_WAREHOUSE: COMPUTE_WH
        EPOCHS: "50"
  $$;

In [ ]:
%%sql -r wait_result
-- Wait for an async job to finish (timeout = 600 seconds)
CALL spcs_demo_db.spcs_schema.my_async_job!SPCS_WAIT_FOR('DONE', 600);

**Parallel batch jobs:** You can run multiple replicas of a job to split work:

```sql
EXECUTE JOB SERVICE
  IN COMPUTE POOL spcs_demo_pool
  NAME = my_parallel_job
  REPLICAS = 10
  FROM SPECIFICATION $$ ... $$;
```

Each replica gets two environment variables:
- `SNOWFLAKE_JOBS_COUNT` = total number of replicas (10)
- `SNOWFLAKE_JOB_INDEX` = this replica's index (0, 1, 2, ... 9)

Your code uses these to partition the work (e.g., replica 0 processes rows 0-999, replica 1 processes rows 1000-1999, etc.).

---
## 5. Service Specification (YAML)

**What:** A YAML document that tells Snowflake everything about how to run your container.

**Why:** It's the blueprint — containers, ports, environment variables, resources, volumes, health checks.

**Two ways to provide it:**
1. **Inline** — directly in SQL between `$$` delimiters
2. **From a stage file** — `FROM @my_stage SPECIFICATION_FILE='spec.yaml'`

**Full specification structure:**

```yaml
spec:
  containers:
  - name: my-container                 # unique name
    image: /db/schema/repo/img:tag     # image from your repository
    command: ["python", "main.py"]     # override entrypoint (optional)
    args: ["--flag=value"]             # arguments (optional)
    env:                               # environment variables
      MY_VAR: "value"
    resources:                         # CPU/memory/GPU
      requests:
        memory: 1G
        cpu: 1
        nvidia.com/gpu: 1              # GPU request
      limits:
        memory: 4G
        cpu: 2
    readinessProbe:                    # health check
      port: 8080
      path: /health
    volumeMounts:                      # attach volumes
    - name: data-vol
      mountPath: /data
    secrets:                           # inject secrets as env vars
    - snowflakeSecret: my_secret
      secretKeyRef: password
      envVarName: DB_PASSWORD

  endpoints:                           # network ports to expose
  - name: api
    port: 8080
    public: false                      # true = internet-accessible

  volumes:                             # storage
  - name: data-vol
    source: block                      # persistent block storage
    size: 10Gi
  - name: stage-vol
    source: "@db.schema.my_stage"     # Snowflake stage mount
  - name: tmp
    source: memory                     # in-memory tmpfs
    size: 256M
```

### Dockerfile vs Service Specification

| | Dockerfile | Service Specification (YAML) |
|---|---|---|
| **Purpose** | How to BUILD the image | How to RUN the image in Snowflake |
| **When it runs** | Once, at build time (your laptop) | Every time the service starts |
| **Contains** | OS, packages, code, startup command | CPU/RAM/GPU, ports, secrets, volumes, replicas |
| **Output** | A static image file | A running container on a compute pool |

**Why both?** A Dockerfile has no concept of Snowflake-specific runtime needs:
- "Give me 4GB RAM and 1 GPU"
- "Mount this Snowflake stage at /data"
- "Inject this Snowflake secret as an env var"
- "Expose port 8080 publicly"
- "Run 3 replicas for load balancing"

**Analogy:** Dockerfile = recipe for baking a cake. Service Spec = instructions for serving it (which table, how many slices, what plates).

**Example showing the split:**

```dockerfile
# Dockerfile (build time — your laptop)
FROM python:3.11-slim
COPY app.py .
RUN pip install flask torch
CMD ["python", "app.py"]
```

```yaml
# Service Spec (run time — tells Snowflake HOW to deploy it)
spec:
  containers:
  - name: model-server
    image: /db/schema/repo/my-app:latest   # the built image
    env:
      MODEL_VERSION: "v2"                  # runtime config
    resources:
      requests:
        nvidia.com/gpu: 1                  # needs GPU
        memory: 8G
    secrets:
    - snowflakeSecret: my_api_key
      secretKeyRef: secret_string
      envVarName: API_KEY
  endpoints:
  - name: api
    port: 8080
    public: true
  volumes:
  - name: models
    source: "@db.schema.model_stage"
```

**Key insight:** Same image, different specs. Deploy with 1 GPU for testing, 4 GPUs for production — no rebuild needed.

---
## 6. Endpoints (Network Ingress)

**What:** Named network ports your service exposes. How the outside world talks to your container.

**Key facts:**
- Defined in the `endpoints` section of the service spec
- Each endpoint has a `name` — referenced by service functions and other services
- `public: true` = accessible from the internet via a Snowflake-managed URL
- `public: false` (default) = only reachable within Snowflake

| Setting | Who Can Access | Use Case |
|---------|---------------|----------|
| `public: false` | Service functions, other SPCS services | ML inference, internal APIs |
| `public: true` | Internet users (with Snowflake auth) | Web UIs, public APIs |

```sql
-- Get the public endpoint URL for a service
SHOW ENDPOINTS IN SERVICE my_service;
-- Look at the 'ingress_url' column
```

The URL looks like: `https://abc123-myorg-myacct.snowflakecomputing.app`

---
## 7. Ingress Gateway

**What:** A Snowflake-managed reverse proxy that sits in front of your service's public endpoints. It handles authentication, HTTPS termination, and request routing for all external (internet-facing) traffic.

**Why:** When you set `public: true` on an endpoint, traffic doesn't hit your container directly. It flows through the Ingress Gateway first — this ensures only authenticated users can reach your service.

**How it works:**

```
External User (browser / API client)
         │
         ▼
┌─────────────────────────────┐
│    INGRESS GATEWAY          │
│  • HTTPS termination (TLS)  │
│  • OAuth / Snowflake auth   │
│  • Routes to correct svc    │
└─────────────────────────────┘
         │
         ▼
┌─────────────────────────────┐
│   Your Service Container    │
│   (port 8080, etc.)         │
└─────────────────────────────┘
```

**Key facts:**
- Automatically provisioned when you create a service with a `public: true` endpoint
- Provides a `*.snowflakecomputing.app` URL (shown in `SHOW ENDPOINTS IN SERVICE`)
- Handles OAuth authentication — users must log into Snowflake before accessing the service
- Supports the `Authorization` header with Snowflake OAuth tokens for programmatic access
- You do NOT create or manage the gateway yourself — Snowflake handles it entirely
- The gateway enforces Snowflake RBAC: only roles with USAGE on the service can access it

**Authentication flow for public endpoints:**

1. User visits `https://abc123-myorg-myacct.snowflakecomputing.app`
2. Gateway redirects to Snowflake login (if not already authenticated)
3. User authenticates with Snowflake credentials
4. Gateway validates the session and checks RBAC (does user's role have USAGE on the service?)
5. If authorized, gateway forwards the request to your container
6. Your container receives the request with additional headers:
   - `Sf-Context-Current-User` — the authenticated Snowflake username
   - `Sf-Context-Current-Role` — the user's active role

**Accessing your public endpoint programmatically:**

```python
import requests

# Get an OAuth token via Snowflake's token endpoint, then:
headers = {
    'Authorization': f'Snowflake Token="{oauth_token}"'
}
response = requests.get(
    'https://abc123-myorg-myacct.snowflakecomputing.app/predict',
    headers=headers
)
```

**Gateway vs no gateway (public vs private endpoints):**

| | `public: false` (default) | `public: true` |
|---|---|---|
| Gateway involved? | No | Yes |
| Reachable from internet? | No | Yes |
| Authentication | None (internal only) | Snowflake OAuth via gateway |
| URL | Internal DNS only | `*.snowflakecomputing.app` |
| Use case | Service functions, inter-service calls | Web UIs, external API clients |

**Important notes:**
- The gateway adds a small latency overhead (authentication check) compared to internal calls
- For high-throughput ML inference called from SQL, prefer `public: false` + service functions (bypasses the gateway entirely)
- The gateway URL becomes available once the service is in `READY` state — check with `SHOW ENDPOINTS IN SERVICE`

---
## 8. Service Functions

**What:** A SQL function (UDF) that sends HTTP requests to your running service. The bridge between SQL and your container.

**Why:** Lets any SQL query call your containerized logic. Users don't need to know about containers — they just call a function.

**How it works:**
1. SQL query calls `my_function(args)`
2. Snowflake batches rows and sends them as HTTP POST to your container
3. Your container processes the batch and returns results
4. Results flow back into the SQL query

**Request format (JSON sent TO your container):**
```json
{
  "data": [
    [0, "first row value"],
    [1, "second row value"]
  ]
}
```

**Response format (JSON your container must RETURN):**
```json
{
  "data": [
    [0, "result for row 0"],
    [1, "result for row 1"]
  ]
}
```

The first element in each inner array is the row index (must match request).

In [ ]:
%%sql -r create_func_result
-- Create a service function
CREATE OR REPLACE FUNCTION spcs_demo_db.spcs_schema.predict_sentiment(text VARCHAR)
  RETURNS VARCHAR
  SERVICE = spcs_demo_db.spcs_schema.echo_service
  ENDPOINT = 'api'           -- must match an endpoint name in the spec
  MAX_BATCH_ROWS = 100       -- rows per HTTP request
  AS '/predict';             -- HTTP path on the container

-- Use the function in a query (just like any built-in function!)
SELECT spcs_demo_db.spcs_schema.predict_sentiment('I love Snowflake!') AS sentiment;

-- Use on a whole table (Snowflake batches automatically)
SELECT review_text, predict_sentiment(review_text) AS sentiment
FROM customer_reviews;

**Service function options:**

| Option | Default | Purpose |
|--------|---------|--------|
| `MAX_BATCH_ROWS` | (all) | Max rows per HTTP request |
| `MAX_BATCH_RETRIES` | 3 | Retry failed batches |
| `ON_BATCH_FAILURE` | ABORT | ABORT or RETURN_NULL |
| `BATCH_TIMEOUT_SECS` | 3600 | Max seconds per batch |
| `CONTEXT_HEADERS` | (none) | Pass Snowflake context (CURRENT_USER, etc.) |
| `VOLATILE` / `IMMUTABLE` | VOLATILE | Caching hint |

---
## 9. External Access Integration (Egress)

**What:** Allows your container to make outbound network calls to the internet (e.g., call an external API, download a file).

**Why:** By default, SPCS containers have **NO internet access** — they can only talk to Snowflake. This is a security feature. You must explicitly allow outbound traffic.

**How it works:**
1. Create a **Network Rule** — defines which domains/IPs are allowed
2. Create an **External Access Integration** — wraps the network rule into a referenceable object
3. Attach it to the service via `EXTERNAL_ACCESS_INTEGRATIONS`

**Key facts:**
- Without this, `requests.get('https://api.openai.com/...')` inside your container will fail
- You control exactly which domains are reachable (allowlist model)
- Can attach multiple integrations to one service

In [ ]:
%%sql -r network_rule_result
-- Step 1: Create a network rule (allowlist specific domains)
CREATE OR REPLACE NETWORK RULE spcs_demo_db.spcs_schema.allow_openai_rule
  TYPE = HOST_PORT
  MODE = EGRESS
  VALUE_LIST = ('api.openai.com:443', 'huggingface.co:443');

-- Step 2: Create the External Access Integration
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION spcs_egress_eai
  ALLOWED_NETWORK_RULES = (spcs_demo_db.spcs_schema.allow_openai_rule)
  ENABLED = TRUE;

-- Step 3: Use it when creating a service
CREATE SERVICE my_service
  IN COMPUTE POOL spcs_demo_pool
  EXTERNAL_ACCESS_INTEGRATIONS = (spcs_egress_eai)
  FROM SPECIFICATION $$ ... $$;

-- Verify integration exists
SHOW EXTERNAL ACCESS INTEGRATIONS;

---
## 10. Volumes (Storage for Containers)

**What:** Storage that containers can read/write to — either ephemeral, persistent, or backed by a Snowflake stage.

**Why:** Containers are stateless by default (data is lost on restart). Volumes give you persistence, shared storage between containers, or access to Snowflake stages.

**Volume types:**

| Source | Persistence | Use Case |
|--------|-------------|----------|
| `block` | Persistent across restarts | ML model checkpoints, databases |
| `memory` | Ephemeral (RAM-backed tmpfs) | Fast scratch space, /tmp replacement |
| `"@db.schema.stage"` | Backed by Snowflake stage | Read input files, write output files |
| `local` | Per-node, shared between containers | Inter-container data exchange |

**Example in spec:**
```yaml
spec:
  containers:
  - name: worker
    image: /db/schema/repo/img:latest
    volumeMounts:
    - name: model-storage
      mountPath: /models        # container sees files here
    - name: input-data
      mountPath: /input
      readOnly: true            # can't modify stage data
    - name: scratch
      mountPath: /tmp

  volumes:
  - name: model-storage
    source: block
    size: 50Gi                  # persistent disk
  - name: input-data
    source: "@my_db.my_schema.training_data_stage"
  - name: scratch
    source: memory
    size: 512M                  # RAM disk
```

**Stage volumes are powerful:** Your container can directly read files from a Snowflake stage as if they were local files — no need to download them first.

---
## 11. Secrets

**What:** Snowflake secret objects that are securely injected into your container as environment variables or files.

**Why:** You should never hardcode credentials (API keys, passwords) in your Docker image. Secrets let you manage them securely in Snowflake and pass them at runtime.

**Types of secrets:**

| Type | Use Case |
|------|----------|
| `GENERIC_STRING` | API keys, tokens, passwords |
| `PASSWORD` | Username/password pairs |
| `OAUTH2` | OAuth2 client credentials |

In [ ]:
%%sql -r secrets_result
-- Create secrets
CREATE OR REPLACE SECRET spcs_demo_db.spcs_schema.my_api_key
  TYPE = GENERIC_STRING
  SECRET_STRING = 'sk-your-api-key-here';

CREATE OR REPLACE SECRET spcs_demo_db.spcs_schema.db_credentials
  TYPE = PASSWORD
  USERNAME = 'admin'
  PASSWORD = 'super-secret-password';

**Using secrets in service spec:**

```yaml
spec:
  containers:
  - name: my-app
    image: /db/schema/repo/img:latest
    secrets:
    - snowflakeSecret: spcs_demo_db.spcs_schema.my_api_key
      secretKeyRef: secret_string    # for GENERIC_STRING
      envVarName: OPENAI_API_KEY     # container sees this env var
    - snowflakeSecret: spcs_demo_db.spcs_schema.db_credentials
      secretKeyRef: username
      envVarName: DB_USER
    - snowflakeSecret: spcs_demo_db.spcs_schema.db_credentials
      secretKeyRef: password
      envVarName: DB_PASS
```

**Inside your container code:**
```python
import os
api_key = os.environ['OPENAI_API_KEY']  # 'sk-your-api-key-here'
db_user = os.environ['DB_USER']         # 'admin'
db_pass = os.environ['DB_PASS']         # 'super-secret-password'
```

**Security notes:**
- Secrets are encrypted at rest in Snowflake
- Only roles with USAGE on the secret can reference it
- Secrets are never written to container logs
- Rotate them with `ALTER SECRET ... SET SECRET_STRING = 'new-value'`

---
## 12. Logging, Monitoring & Event Table

**What:** SPCS containers write stdout/stderr to an **Event Table** — a system table that stores logs, metrics, and traces.

**Why:** You can't SSH into containers. Logs are your only debugging tool. The Event Table is where they go.

**How it works:**
- Anything your container prints to stdout/stderr is captured
- Snowflake stores it in the account's Event Table
- You query it with SQL

**Setup:** Your account needs an event table configured (usually done by ACCOUNTADMIN once).

In [ ]:
%%sql -r event_table_check
-- Check if your account has an event table set
SHOW PARAMETERS LIKE 'EVENT_TABLE' IN ACCOUNT;

-- View container logs for a specific service
SELECT 
  TIMESTAMP,
  RESOURCE_ATTRIBUTES['snow.service.name']::STRING AS service_name,
  RESOURCE_ATTRIBUTES['snow.service.container.name']::STRING AS container_name,
  RECORD['severity_text']::STRING AS log_level,
  VALUE::STRING AS log_message
FROM EVENT_DB.EVENT_SCHEMA.EVENT_TABLE -- your actual event table
WHERE RESOURCE_ATTRIBUTES['snow.service.name'] = 'ECHO_SERVICE'
  AND TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
ORDER BY TIMESTAMP DESC
LIMIT 50;

-- Quick service health check commands
-- Get service status
SELECT SYSTEM$GET_SERVICE_STATUS('spcs_demo_db.spcs_schema.echo_service');

-- Get service logs (shortcut — last few lines of stdout/stderr)
CALL SYSTEM$GET_SERVICE_LOGS('spcs_demo_db.spcs_schema.echo_service', '0', 'echo', 50);

**Debugging workflow:**
1. `DESCRIBE SERVICE` — is it running? check status
2. `SYSTEM$GET_SERVICE_STATUS()` — detailed JSON status per container
3. `SYSTEM$GET_SERVICE_LOGS()` — read last N lines of stdout/stderr
4. Event Table query — full log history with filtering

**SYSTEM$GET_SERVICE_LOGS parameters:**
```sql
CALL SYSTEM$GET_SERVICE_LOGS(
  'db.schema.service_name',  -- fully qualified service name
  '0',                       -- instance ID (usually '0')
  'container_name',          -- container name from spec
  50                         -- number of lines (most recent)
);
```

---
## 13. Service-to-Service Communication

**What:** SPCS services can call each other over the internal network without going through the public internet.

**Why:** Microservice architectures — e.g., a web frontend calls an ML backend.

**How it works:**
- Every service gets an internal DNS name: `<service_name>.<schema>.<db>`
- Services in the same account can reach each other's endpoints directly
- No External Access Integration needed for this — it's all internal

**Example:**
If `echo_service` has an endpoint named `api` on port 8080, another SPCS service can reach it at:
```
http://echo_service.spcs_schema.spcs_demo_db:8080/predict
```

**From your container code:**
```python
import requests
# Call another SPCS service directly
response = requests.post(
    'http://echo_service.spcs_schema.spcs_demo_db:8080/predict',
    json={'data': [[0, 'hello']]}
)
```

**Requirements:**
- Both services must be in the same Snowflake account
- The calling role must have USAGE on the target service
- The target endpoint must NOT be marked `public: true` (internal-only endpoints are best for this)

---
## 14. Security & Roles (Privileges)

**What:** SPCS follows Snowflake's RBAC model. Each component requires specific privileges.

**Why:** You need to grant the right privileges to the role creating/using services. Without them, you'll get access denied errors.

**Required privileges by component:**

| Action | Required Privilege |
|--------|-------------------|
| Create compute pool | CREATE COMPUTE POOL ON ACCOUNT |
| Use compute pool | USAGE or OPERATE ON COMPUTE POOL |
| Create service | CREATE SERVICE ON SCHEMA |
| Use service function | USAGE ON SERVICE + function grants |
| Create image repo | CREATE IMAGE REPOSITORY ON SCHEMA |
| Push images | WRITE ON IMAGE REPOSITORY |
| Use secrets | USAGE ON SECRET |
| Bind EAI to service | USAGE ON INTEGRATION |

In [ ]:
%%sql -r grant_result
-- Grant a role the ability to create and use SPCS resources
-- (Run as ACCOUNTADMIN or a role with MANAGE GRANTS)

-- Allow role to create compute pools
GRANT CREATE COMPUTE POOL ON ACCOUNT TO ROLE spcs_developer_role;

-- Allow role to create services in a schema
GRANT CREATE SERVICE ON SCHEMA spcs_demo_db.spcs_schema TO ROLE spcs_developer_role;

-- Allow role to use a compute pool
GRANT USAGE, OPERATE ON COMPUTE POOL spcs_demo_pool TO ROLE spcs_developer_role;

-- Allow role to use the external access integration
GRANT USAGE ON INTEGRATION spcs_egress_eai TO ROLE spcs_developer_role;

-- Allow role to read/write to the image repository
GRANT READ, WRITE ON IMAGE REPOSITORY spcs_demo_db.spcs_schema.my_repo TO ROLE spcs_developer_role;

**Snowflake-provided credentials inside containers:**

Every SPCS container automatically gets a Snowflake session token (like being "logged in" as the owner role). Your code can connect to Snowflake without any manual credential setup:

```python
import snowflake.connector
import os

# These are auto-injected by SPCS into every container
conn = snowflake.connector.connect(
    host=os.environ['SNOWFLAKE_HOST'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    token=open('/snowflake/session/token').read(),
    authenticator='oauth'
)

cursor = conn.cursor()
cursor.execute('SELECT CURRENT_USER(), CURRENT_ROLE()')
print(cursor.fetchone())  # shows the service's owner role
```

This means your container can read/write Snowflake tables, stages, etc. — governed by the service owner role's privileges.

---
## 15. Complete End-to-End Example (Summary)

Here's the full workflow from zero to a working service function:

```
1. CREATE IMAGE REPOSITORY         → Store for your Docker images
         ↓
2. docker build + docker push      → Push image to Snowflake (from local machine)
         ↓
3. CREATE COMPUTE POOL             → Provision VMs to run containers
         ↓
4. (Optional) CREATE SECRET        → For API keys / passwords
         ↓
5. (Optional) CREATE EAI           → If container needs internet
         ↓
6. CREATE SERVICE                  → Start the container
         ↓
7. DESCRIBE SERVICE / GET LOGS     → Verify it's healthy
         ↓
8. CREATE FUNCTION ... SERVICE =   → Create SQL-callable function
         ↓
9. SELECT my_function(col)         → Use in any SQL query!
```

**Minimum viable example (for the impatient):**
```sql
-- All you truly need:
CREATE COMPUTE POOL my_pool MIN_NODES=1 MAX_NODES=1 INSTANCE_FAMILY=CPU_X64_XS;
CREATE IMAGE REPOSITORY my_db.my_schema.my_repo;
-- (push image from terminal)
CREATE SERVICE my_service IN COMPUTE POOL my_pool FROM SPECIFICATION $$ ... $$;
CREATE FUNCTION my_func(x VARCHAR) RETURNS VARCHAR SERVICE=my_service ENDPOINT='ep' AS '/path';
SELECT my_func('hello');
```

---
## 16. Cleanup (Drop Resources)

**Important:** Resources cost credits while they exist. Always clean up when done experimenting.

**Order matters!** Drop in reverse creation order — services first, then compute pool.

In [ ]:
%%sql -r cleanup_result
-- CLEANUP: Run these to tear down everything created in this notebook
-- (Uncomment the lines you want to execute)

-- 1. Drop service function
-- DROP FUNCTION IF EXISTS spcs_demo_db.spcs_schema.predict_sentiment(VARCHAR);

-- 2. Drop services
-- DROP SERVICE IF EXISTS spcs_demo_db.spcs_schema.echo_service;

-- 3. Drop compute pool (must have no services)
-- ALTER COMPUTE POOL spcs_demo_pool STOP ALL;  -- force-stop all services
-- DROP COMPUTE POOL IF EXISTS spcs_demo_pool;

-- 4. Drop supporting objects
-- DROP SECRET IF EXISTS spcs_demo_db.spcs_schema.my_api_key;
-- DROP SECRET IF EXISTS spcs_demo_db.spcs_schema.db_credentials;
-- DROP EXTERNAL ACCESS INTEGRATION IF EXISTS spcs_egress_eai;
-- DROP IMAGE REPOSITORY IF EXISTS spcs_demo_db.spcs_schema.my_repo;

-- 5. Drop schema/database (if you want a full reset)
-- DROP SCHEMA IF EXISTS spcs_demo_db.spcs_schema;
-- DROP DATABASE IF EXISTS spcs_demo_db;

---
## Quick Reference Card

| Command | Purpose |
|---------|--------|
| `SHOW IMAGE REPOSITORIES` | List repos |
| `SHOW IMAGES IN IMAGE REPOSITORY repo` | List pushed images |
| `SHOW COMPUTE POOLS` | List compute pools |
| `DESCRIBE COMPUTE POOL pool` | Pool status |
| `SHOW SERVICES` | List services |
| `DESCRIBE SERVICE svc` | Service status |
| `SHOW ENDPOINTS IN SERVICE svc` | Get endpoint URLs |
| `SHOW SERVICE CONTAINERS IN SERVICE svc` | Container status |
| `SYSTEM$GET_SERVICE_STATUS('svc')` | JSON status |
| `SYSTEM$GET_SERVICE_LOGS('svc','0','container',N)` | Read logs |
| `ALTER COMPUTE POOL pool SUSPEND` | Stop billing |
| `ALTER SERVICE svc SUSPEND` | Pause service |
| `ALTER SERVICE svc RESUME` | Restart service |

---

**You now have a complete understanding of SPCS!** Work through the SQL cells top-to-bottom to deploy your first service.

### API Integration vs External Access Integration

These are two **completely different objects** that solve different problems. The naming is confusing, but they are not interchangeable.

| | API Integration | External Access Integration (EAI) |
|---|---|---|
| **Purpose** | Let Snowflake RECEIVE calls or call external APIs on behalf of SQL | Let YOUR CODE (UDF/procedure/container) make outbound HTTP calls |
| **Direction** | Snowflake ↔ external service (managed by Snowflake) | Your code → internet (you write the HTTP call) |
| **Used by** | External functions, API integrations with cloud services | Python/Java UDFs, stored procedures, SPCS containers |
| **Who makes the HTTP call?** | Snowflake's infrastructure | Your code (e.g., `requests.get(...)`) |
| **Created with** | `CREATE API INTEGRATION` | `CREATE EXTERNAL ACCESS INTEGRATION` |

---

**API Integration — Snowflake calls an external API FOR you**

Use when: You want to create an External Function that routes SQL calls to an API Gateway (AWS API Gateway, Azure API Management, GCP API Gateway).

```sql
-- Scenario: You have a Lambda function behind AWS API Gateway.
-- You want to call it from SQL like: SELECT my_lambda_func(col)

-- Step 1: Create API Integration (tells Snowflake about the gateway)
CREATE OR REPLACE API INTEGRATION my_aws_api_integration
  API_PROVIDER = aws_api_gateway
  API_AWS_ROLE_ARN = 'arn:aws:iam::123456789:role/my-sf-role'
  API_ALLOWED_PREFIXES = ('https://abc123.execute-api.us-east-1.amazonaws.com/')
  ENABLED = TRUE;

-- Step 2: Create External Function (SQL function that calls the API)
CREATE OR REPLACE EXTERNAL FUNCTION my_lambda_func(input VARCHAR)
  RETURNS VARCHAR
  API_INTEGRATION = my_aws_api_integration
  AS 'https://abc123.execute-api.us-east-1.amazonaws.com/prod/my-endpoint';

-- Step 3: Use it
SELECT my_lambda_func('hello');
```

**You never write HTTP code.** Snowflake handles the HTTP call, authentication, and response parsing.

---

**External Access Integration (EAI) — YOUR code calls the internet**

Use when: Your Python UDF, stored procedure, or SPCS container needs to make outbound HTTP requests directly (e.g., call OpenAI, download a file, hit a REST API).

```sql
-- Scenario: Your Python UDF needs to call the OpenAI API directly.

-- Step 1: Network Rule (allowlist specific domains)
CREATE OR REPLACE NETWORK RULE allow_openai
  TYPE = HOST_PORT
  MODE = EGRESS
  VALUE_LIST = ('api.openai.com:443');

-- Step 2: External Access Integration (wraps the rule)
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION openai_eai
  ALLOWED_NETWORK_RULES = (allow_openai)
  ENABLED = TRUE;

-- Step 3: Use it in a UDF (YOUR code makes the HTTP call)
CREATE OR REPLACE FUNCTION call_openai(prompt VARCHAR)
  RETURNS VARCHAR
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.11'
  PACKAGES = ('requests')
  EXTERNAL_ACCESS_INTEGRATIONS = (openai_eai)
  HANDLER = 'run'
AS $$
import requests, os
def run(prompt):
    resp = requests.post('https://api.openai.com/v1/chat/completions',
        headers={'Authorization': f'Bearer {os.environ["API_KEY"]}'},
        json={'model': 'gpt-4', 'messages': [{'role': 'user', 'content': prompt}]})
    return resp.json()['choices'][0]['message']['content']
$$;

-- For SPCS: attach EAI to a service
-- CREATE SERVICE my_svc
--   IN COMPUTE POOL my_pool
--   EXTERNAL_ACCESS_INTEGRATIONS = (openai_eai)
--   FROM SPECIFICATION $$ ... $$;
```

---

**Decision flowchart:**

```
Do you want to call an external API from SQL?
  ├── YES, and I don't want to write HTTP code
  │     → API Integration + External Function
  │       (Snowflake handles the HTTP call via API Gateway)
  │
  └── YES, and my Python/Java code needs direct internet access
        → External Access Integration + Network Rule
          (Your code does requests.get/post, Snowflake just opens the door)
```

**In short:**
- **API Integration** = Snowflake is the HTTP client (you just call a SQL function)
- **EAI** = Your code is the HTTP client (Snowflake just allows the outbound connection)